# Day 74 - Pandas 数据清洗

本节内容覆盖两大核心主题：**缺失值处理** 和 **重复值处理**。

实际工作中，从 Excel、CSV 或数据库获取的数据往往不完美：可能存在因系统或人为原因产生的缺失值、重复值或异常值，也存在格式不统一等问题。在开始数据分析之前，对数据进行清洗非常关键。

In [ ]:
import numpy as np
import pandas as pd

## 1. 准备示例数据

先构造一个员工表和一个部门表，供后续演示使用。

In [ ]:
# 员工表
emp_df = pd.DataFrame(
    data={
        'ename': ['胡一刀', '乔峰', '李莫愁', '张无忌', '丘处机',
                   '欧阳锋', '张翠山', '黄蓉', '杨过', '朱九真',
                   '苗人凤', '郭靖', '宋远桥', '张三丰'],
        'job': ['销售员', '分析师', '设计师', '程序员', '程序员',
                '程序员', '程序员', '销售主管', '会计', '会计',
                '销售员', '出纳', '会计师', '总裁'],
        'mgr': [3344, 7800, 2056, 2056, 2056, 3088, 2056, 7800,
                5566, 5566, 3344, 5566, 7800, np.nan],
        'sal': [1800, 5000, 3500, 3200, 3400, 3200, 4000, 3000,
                2200, 2500, 2500, 2000, 4000, 9000],
        'comm': [200, 1500, 800, np.nan, np.nan, np.nan, np.nan, 800,
                 np.nan, np.nan, np.nan, np.nan, 1000, 1200],
        'dno': [30, 20, 20, 20, 20, 20, 20, 30, 10, 10, 30, 10, 10, 20]
    },
    index=[1359, 2056, 3088, 3211, 3233, 3244, 3251, 3344, 3577, 3588, 4466, 5234, 5566, 7800]
)
emp_df.index.name = 'eno'
emp_df

In [ ]:
# 部门表
dept_df = pd.DataFrame(
    data={
        'dname': ['会计部', '研发部', '销售部', '运维部'],
        'dloc': ['北京', '成都', '重庆', '天津']
    },
    index=[10, 20, 30, 40]
)
dept_df.index.name = 'dno'
dept_df

---
## 2. 缺失值处理

### 2.1 检测缺失值

使用 `isnull()` 或 `isna()` 方法可以检测 DataFrame 中的缺失值，返回一个布尔值 DataFrame，缺失位置为 `True`。

In [ ]:
# 检测缺失值
emp_df.isnull()

In [ ]:
# 统计每列缺失值的数量
emp_df.isnull().sum()

In [ ]:
# 统计整体缺失值数量
emp_df.isnull().sum().sum()

与 `isnull()` / `isna()` 相对的是 `notnull()` / `notna()`，它们将非空值标记为 `True`。

In [ ]:
# notna() 返回非缺失值的布尔标记
emp_df.notna()

### 2.2 删除缺失值 —— `dropna()`

使用 `dropna()` 方法可以删除包含缺失值的行或列：
- `axis=0`（默认）：沿行方向删除，即遇到空值删除整行
- `axis=1`：沿列方向删除，即遇到空值删除整列

In [ ]:
# 删除包含缺失值的行（默认 axis=0）
emp_df.dropna()

In [ ]:
# 删除包含缺失值的列
emp_df.dropna(axis=1)

> **注意**：`dropna()` 默认不会修改原 DataFrame，而是返回一个新对象。如果想直接在原数据上修改，可以设置 `inplace=True`。

### 2.3 填充缺失值 —— `fillna()`

使用 `fillna()` 方法可以对缺失值进行填充：
- `value` 参数：指定用某个值填充
- `method='ffill'`：用前一个非空值填充
- `method='bfill'`：用后一个非空值填充

In [ ]:
# 用 0 填充所有缺失值
emp_df.fillna(value=0)

In [ ]:
# 用列的均值填充
emp_df.fillna(value=emp_df.mean(numeric_only=True))

In [ ]:
# 前向填充：用前一个非空值填充
emp_df.fillna(method='ffill')

In [ ]:
# 后向填充：用后一个非空值填充
emp_df.fillna(method='bfill')

> **注意**：填充策略的选择值得深入探讨。实际工作中可能使用均值、众数等统计量，或随机插值法、拉格朗日插值法，甚至回归模型、贝叶斯模型来填充缺失数据。

---
## 3. 重复值处理

### 3.1 构造含重复数据的示例

先给部门表添加几行重复数据，让研发部和销售部各出现两次。

In [ ]:
# 给部门表添加重复数据
dept_dup = dept_df.copy()
dept_dup.loc[50] = {'dname': '研发部', 'dloc': '上海'}
dept_dup.loc[60] = {'dname': '销售部', 'dloc': '长沙'}
dept_dup

### 3.2 检测重复值 —— `duplicated()`

`duplicated()` 方法用于判断是否存在重复值：
- 不指定参数时默认按整行判断
- 可以指定一个或多个列名，按指定列判断重复

In [ ]:
# 按部门名称判断重复
dept_dup.duplicated('dname')

In [ ]:
# 查看哪些行是重复的（所有重复项都显示）
dept_dup[dept_dup.duplicated('dname', keep=False)]

### 3.3 删除重复值 —— `drop_duplicates()`

`drop_duplicates()` 方法用于删除重复值：
- `keep='first'`（默认）：保留第一次出现的记录
- `keep='last'`：保留最后一次出现的记录
- `keep=False`：删除所有重复项

In [ ]:
# 按 dname 去重，保留第一条
dept_dup.drop_duplicates('dname')

In [ ]:
# 按 dname 去重，保留最后一条
dept_dup.drop_duplicates('dname', keep='last')

### 3.4 对员工数据去重

假设我们认为 ename 和 job 两个字段完全相同的就是重复数据。先构造含重复记录的员工表，再进行去重。

In [ ]:
# 构造含重复员工数据的 DataFrame
emp2_df = pd.DataFrame(
    data={
        'ename': ['张三丰', '王大锤', '张三丰', '骆昊', '陈小刀'],
        'job': ['总裁', '程序员', '总裁', '架构师', '分析师'],
        'mgr': [np.nan, 9800, np.nan, 7800, 9800],
        'sal': [50000, 8000, 60000, 30000, 10000],
        'comm': [8000, 600, 6000, 5000, 1200],
        'dno': [20, 20, 20, 20, 20]
    },
    index=[9500, 9600, 9700, 9800, 9900]
)
emp2_df.index.name = 'eno'

# 拼接
all_emp_df = pd.concat([emp_df, emp2_df])
all_emp_df

In [ ]:
# 查看 ename 和 job 完全相同的重复记录
all_emp_df[all_emp_df.duplicated(['ename', 'job'], keep=False)]

In [ ]:
# 按 ename 和 job 去重
all_emp_df.drop_duplicates(['ename', 'job'], inplace=True)
all_emp_df

---
## 4. 综合练习

下面把缺失值和重复值处理串联起来，模拟一个典型的数据清洗流程。

In [ ]:
# 1) 构造一份带有缺失值和重复行的数据
dirty_df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie', 'Alice', 'David', None, 'Bob'],
    'age': [25, 30, np.nan, 25, 40, 35, 30],
    'score': [88, np.nan, 72, 88, 95, 60, np.nan],
    'city': ['Beijing', 'Shanghai', 'Guangzhou', 'Beijing', 'Shenzhen', 'Shanghai', 'Shanghai']
})
dirty_df

In [ ]:
# 2) 查看缺失值概况
print('=== 缺失值统计 ===')
print(dirty_df.isnull().sum())
print(f'\n总缺失值: {dirty_df.isnull().sum().sum()}')

In [ ]:
# 3) 删除重复行
clean_df = dirty_df.drop_duplicates()
clean_df

In [ ]:
# 4) 用列均值填充数值列的缺失值
clean_df['age'] = clean_df['age'].fillna(clean_df['age'].mean())
clean_df['score'] = clean_df['score'].fillna(clean_df['score'].mean())
clean_df

In [ ]:
# 5) 确认清洗后无缺失值
print('=== 清洗后缺失值统计 ===')
print(clean_df.isnull().sum())

---
## 5. 小结

### 缺失值
| 操作 | 方法 | 常用参数 |
|------|------|----------|
| 检测 | `isnull()` / `isna()` | 无 |
| 删除 | `dropna()` | `axis`, `inplace` |
| 填充 | `fillna()` | `value`, `method` (ffill / bfill) |

### 重复值
| 操作 | 方法 | 常用参数 |
|------|------|----------|
| 检测 | `duplicated()` | `subset`, `keep` |
| 删除 | `drop_duplicates()` | `subset`, `keep`, `inplace` |